In [ ]:
%pip install transformers
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu118
%pip install tqdm
%pip install azure-ai-ml azure-identity azure-keyvault-secrets

In [ ]:
!nvidia-smi

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import  os

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)
keyvault_name = ml_client.workspaces.get(ml_client.workspace_name).key_vault.split('/')[-1]
kv_url = f"https://{keyvault_name}.vault.azure.net/"

def readSecret(
    key_vault_url: str,
    secret_name: str
) -> str:
    """
    read secret in keyvault .

    Args:
        key_vault_url:  Key Vault url.
        secret_name:    Name of the secret
    Returns:
        secret value (str).
    """        
    kv_client = SecretClient(vault_url=key_vault_url, credential=credential)
    secret_value = kv_client.get_secret(secret_name).value
    return secret_value

global_model_id = readSecret(kv_url,"AZURE-ML-MODEL-ID")
global_model_id=os.getenv("AZURE_ML_MODEL_ID",default=global_model_id)
print(f"global_model_id:{global_model_id}")

global_model_name = readSecret(kv_url,"AZURE-ML-MODEL-NAME")
global_model_name=os.getenv("AZURE_ML_MODEL_NAME",default=global_model_name)
print(f"global_model_name:{global_model_name}")

global_endpoint_name = readSecret(kv_url,"AZURE-ML-ENDPOINT-NAME")
global_endpoint_name=os.getenv("AZURE_ML_ENDPOINT_NAME",default=global_endpoint_name)
print(f"global_endpoint_name:{global_endpoint_name}")

global_env_name = readSecret(kv_url,"AZURE-ML-ENV-NAME")
global_env_name=os.getenv("AZURE_ML_ENV_NAME",default=global_env_name)
print(f"global_env_name:{global_env_name}")

global_instance_type = readSecret(kv_url,"AZURE-ML-INSTANCE-TYPE")
global_instance_type=os.getenv("AZURE_ML_INSTANCE_TYPE",default=global_instance_type)
print(f"global_instance_type:{global_instance_type}")

global_score_script = readSecret(kv_url,"AZURE-ML-SCORING-SCRIPT")
global_score_script=os.getenv("AZURE_ML_SCORING_SCRIPT",default=global_score_script)
print(f"global_score_script:{global_score_script}")

global_subscription_id = readSecret(kv_url,"AZURE-ML-SUBSCRIPTION-ID")
global_subscription_id=os.getenv("AZURE_ML_SUBSCRIPTION_ID",default=global_subscription_id)
print(f"global_subscription_id:{global_subscription_id}")
    
global_resource_group_name = readSecret(kv_url,"AZURE-ML-RESOURCE-GROUP")
global_resource_group_name=os.getenv("AZURE_ML_RESOURCE_GROUP",default=global_resource_group_name)
print(f"global_resource_group_name:{global_resource_group_name}")

global_workspace_name = readSecret(kv_url,"AZURE-ML-WORKSPACE-NAME")
global_workspace_name=os.getenv("AZURE_ML_WORKSPACE_NAME",default=global_workspace_name)
print(f"global_workspace_name:{global_workspace_name}")

In [ ]:
from transformers import AutoModel, AutoTokenizer

model_id = global_model_id
model = AutoModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

In [ ]:

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential

subscription_id = global_subscription_id
resource_group = global_resource_group_name
workspace_name = global_workspace_name

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

model = Model(
    name=global_model_name,
    path="./model",
    type="custom_model",
)

model = ml_client.models.create_or_update(model)


In [ ]:

from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

endpoint_name = f"{global_endpoint_name}-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key",  # or "aad_token"
    description=f"{global_model_name} real-time inference endpoint",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()


In [ ]:
import json
import torch
import os
import logging
from transformers import AutoTokenizer, OPTForCausalLM, AutoModelForCausalLM

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

def init():
    global tokenizer, model

    # base = os.environ["AZUREML_MODEL_DIR"]              # e.g. /var/azureml-app/azureml-models/.../1
    # model_dir = os.path.join(base, "model")             # <- because your files are in .../1/model/
    model_dir = "./model"             # <- because your files are in .../1/model/

    #logger.info(f"AZUREML_MODEL_DIR={base}")
    logger.info(f"Using model_dir={model_dir}")
    logger.info(f"Files in model_dir: {os.listdir(model_dir)}")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    # model = AutoModelForCausalLM.from_pretrained(model_dir, local_files_only=True)
    model = OPTForCausalLM.from_pretrained(model_dir, local_files_only=True)
    model.eval()

# def init():
#    global model, tokenizer
#    model_dir = "./model"  # Azure mounts model here automatically
#    tokenizer = AutoTokenizer.from_pretrained(model_dir)
#    model = OPTForCausalLM.from_pretrained(model_dir)
#    model.eval()

def run(raw_data):
    logger.info(f"raw_data={raw_data}")
    data = json.loads(raw_data)
    inputs = tokenizer(data["text"], return_tensors="pt")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        )
    logger.info(f"generated_text={result}")

    return {
        "generated_text": result
    }
    
init()
input_data = "{\"text\": \"The future of artificial intelligence is\"}"
print(f"input_data: {input_data}")
result = run(input_data)
print(f"result: {result}")

In [ ]:
import yaml
from azure.ai.ml.entities import Environment

conda_spec = yaml.safe_load(f"""
name: {global_env_name}
channels:
- conda-forge
dependencies:
- python=3.10
- pip
- pip:
  - torch
  - transformers
  - azureml-inference-server-http
""")

env = Environment(
    name=global_env_name,
    description="Torch + Transformers on minimal inference image",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file=conda_spec,
)

registered_env = ml_client.environments.create_or_update(env)
print("Registered:", registered_env.name, registered_env.version)

# List all versions
for e in ml_client.environments.list(name=global_env_name):
  print(e.name, e.version)

# Or get the one you just created
# ml_client.environments.get(name=registered_env.name, version=registered_env.version)


In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model, CodeConfiguration, ProbeSettings
print(f"endpoint: {endpoint_name}")
print(f"env:{registered_env.name}:{registered_env.version}")
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,                 # or "hf-model:1"
    environment=f"{registered_env.name}:{registered_env.version}",  # or "hf-env@latest"
    code_configuration=CodeConfiguration(code="./src", scoring_script=global_score_script),
    instance_type=global_instance_type,  # e.g. "Standard_DS3_v2"
    instance_count=1,    
    environment_variables={
        "AZUREML_HTTP_PORT": "5001"
    },
    liveness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),
    readiness_probe=ProbeSettings(
        initial_delay=300,
        timeout=10,
        period=10,
        failure_threshold=6,
    ),


)

ml_client.online_deployments.begin_create_or_update(deployment).result()


In [ ]:

# Fetch endpoint credentials
# creds = ml_client.online_endpoints.get_keys(
#     name=endpoint_name
# )

# API keys (for auth_mode: key)
# primary_key = creds.primary_key
#secondary_key = creds.secondary_key

# print("Primary key:", primary_key)
# print("Secondary key:", secondary_key)

# 6) Route traffic to the deployment
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# az ml online-deployment show   --name blue   --endpoint-name hf-opt-endpoint-20260324145548   --resource-group ces3-posttraining   --workspace-name testwest   --query environmentVariables
# az ml online-deployment update   --name blue   --endpoint-name hf-opt-endpoint-20260324145548   --resource-group ces3-posttraining   --workspace-name testwest   --set environment_variables.AZUREML_HTTP_PORT=5001
# az ml online-deployment get-logs   --endpoint-name hf-opt-endpoint-20260324134504   --name blue  --lines 200  --resource-group ces3-posttraining --workspace-name  testwest
# TOKEN=$(az ml online-endpoint get-credentials   --name hf-opt-endpoint-20260324153405   --resource-group ces3-posttraining   --workspace-name testwest --query accessToken -o tsv)
# curl -X POST "https://hf-opt-endpoint-20260324153405.westus.inference.ml.azure.com/score"     -H "Content-Type: application/json"     -H "Authorization: Bearer $TOKEN"     -d '{"text": "I am going to Paris, what should I see?"}'
